# TypedDict

In [13]:
from typing import TypedDict, Annotated, Optional, Literal
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
class Person(TypedDict):
    name: str
    age: int
    
new_person: Person= {'name':'ABCD', 'age': 123} # hover over name or age for type suggestion (it will not throw error even if the type is incorrect)
print(new_person)

{'name': 'ABCD', 'age': 123}


In [4]:
model= ChatMistralAI(model="mistral-medium-2508")

In [ ]:
# Structured Output
class review(TypedDict):
    summary: str
    sentiment: str
    
structured_model= model.with_structured_output(review)

result= structured_model.invoke("The new Apex Phone 15 Pro offers an incredible, bright display and lightning-fast performance that handles heavy gaming with ease. While the battery life lasts all day, the lack of a charger in the box and the premium price tag are minor downsides to an otherwise stellar device.")

print(result)

{'summary': 'The new Apex Phone 15 Pro offers an incredible, bright display and lightning-fast performance that handles heavy gaming with ease. The battery life lasts all day, but the lack of a charger in the box and the premium price tag are minor downsides.', 'sentiment': 'positive'}


In [ ]:
# Structured Output with annotation
class review(TypedDict):
    summary: Annotated[str, "A one sentence summary of the review"] # provides a little guidance to the model
    sentiment: str
    
structured_model= model.with_structured_output(review)

result= structured_model.invoke("The new Apex Phone 15 Pro offers an incredible, bright display and lightning-fast performance that handles heavy gaming with ease. While the battery life lasts all day, the lack of a charger in the box and the premium price tag are minor downsides to an otherwise stellar device.")

print(result)

{'summary': 'The Apex Phone 15 Pro impresses with its incredible display and lightning-fast performance, though it lacks a charger and comes with a premium price tag.', 'sentiment': 'positive'}


In [ ]:
class review(TypedDict): #schema
    
    summary: Annotated[str, "A one sentence summary of the review"] # annotated provides a little guidance to the model
    sentiment: Annotated[Literal['achha', 'bura'], "return the sentiment of the review"]
    key_themes: Annotated[list[str], "Write all the key themes discussed in the review in a list."]
    pros: Annotated[Optional[list[str]], "Write all the pros in a list"]
    cons: Annotated[Optional[list[str]], "Write all the cons in a list"]
    overall_rating: Annotated[int, "On a scale of 1 to 10, write the overall rating of the product"]
    
structured_model= model.with_structured_output(review)

review= """ 
After using the Nova X10 Pro as my primary device for the past two weeks, it is clear that the mid-range smartphone market has a new top contender. Priced at $599, the X10 Pro promises high-end features without the flagship price tag. It generally delivers on that promise, boasting an incredible display and lightning-fast performance, but it isn't perfect.
Here is my in-depth review of the Nova X10 Pro.
Design and Build Quality: Sleek but Slippery
The X10 Pro features a matte glass back that resists fingerprints, surrounded by an aluminum frame that feels very premium in hand. It is surprisingly light at 185g, but the matte finish makes it incredibly slippery—I dropped it twice in the first week.
Pros: IP68 water/dust resistance, satisfying clicky power button.
Cons: Very slippery, prone to scratches on the camera bump.
Display: A Visual Treat
The standout feature is the 6.7-inch AMOLED screen with a 120Hz adaptive refresh rate. Colors are vibrant, and the brightness peaks at 2,000 nits, making it perfect for outdoor viewing in direct sunlight. Watching HDR content on this phone is a joy, as the bezels are almost non-existent.
Performance: Gaming Powerhouse
Powered by the Snapdragon 8 Gen 4 chipset and 12GB of RAM, this phone flies. I ran Genshin Impact at max settings with minimal lag or overheating, thanks to the improved internal vapor chamber cooling system. Multitasking is smooth, and apps stay loaded in the background for a long time.
Camera Performance: Great Daytime, Decent Nighttime
The 50MP main sensor produces clean, sharp, and color-accurate photos in daylight. The portrait mode does an excellent job with edge detection.
However, in low-light situations, the noise reduction kicks in too heavily, leading to a "watercolor" effect on fine details. The 8MP ultrawide is decent but lacks the sharpness of the main lens.
Verdict: Excellent for social media, average for serious photography.
Battery Life and Charging
The 5,000mAh battery consistently lasted me through a full day of heavy usage, usually leaving me with 20% by bedtime. The 100W fast charging is phenomenal, getting me from 0% to 100% in just under 25 minutes.
"""

result= structured_model.invoke(review)

print(result)
print(result['key_themes'])
print(result['pros'])
print(result['cons'])
print(result['sentiment'])
print(result['overall_rating'])

{'summary': 'The Nova X10 Pro is a strong mid-range contender with high-end features like a stunning display, fast performance, and excellent battery life, though it has some drawbacks like a slippery design and average low-light camera performance.', 'sentiment': 'achha', 'key_themes': ['Design and Build Quality', 'Display', 'Performance', 'Camera Performance', 'Battery Life and Charging'], 'pros': ['Premium matte glass and aluminum design', 'IP68 water/dust resistance', '6.7-inch AMOLED display with 120Hz adaptive refresh rate', 'Brightness peaks at 2,000 nits for outdoor visibility', 'Snapdragon 8 Gen 4 chipset and 12GB RAM for smooth performance', 'Excellent cooling system for gaming', '50MP main camera with great daytime performance', '5,000mAh battery with all-day usage', '100W fast charging (0% to 100% in 25 minutes)'], 'cons': ['Slippery design prone to drops', 'Camera bump susceptible to scratches', "Low-light camera performance has a 'watercolor' effect", '8MP ultrawide lacks

# Pydantic

In [ ]:
from pydantic import BaseModel, EmailStr, Field
from typing import Optional
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
class Student(BaseModel): #schema
    
    name: str= 'Fenil' # Pydantic will throw error if data other than str provided
    age: Optional[int]= None # Pydantic will convert into int if number in string format is provided (eg: '99' -> 99) [This is called type coercing]
    email: EmailStr= 'name@domain.com' # Pydantic in-built validation
    cgpa: float= Field(gt=0, le=10, default=5, description='A value representing grades of student') # Pydantic function 'Field' to set default values, constraints, decription, regex expressions, etc.

new_student = {'age': '99'} # keep empty to check default values

student= Student(**new_student) # here we are converting dict to pydantic model.

print(student, "|", type(student))
print(new_student, "|", type(new_student))

student_dict= student.model_dump() # here we are converting pydantic model back to dict (old .dict() is depricated)
print(student_dict, "|", type(student_dict))

student_json= student.model_dump_json() # here we are converting pydantic modelto json (old .json() is depricated)
print(student_json, "|", type(student_json))

name='Fenil' age=99 email='name@domain.com' cgpa=5 | <class '__main__.Student'>
{'age': '99'} | <class 'dict'>
{'name': 'Fenil', 'age': 99, 'email': 'name@domain.com', 'cgpa': 5} | <class 'dict'>
{"name":"Fenil","age":99,"email":"name@domain.com","cgpa":5.0} | <class 'str'>


# Structured Output using Pydantic

In [ ]:
from pydantic import BaseModel, EmailStr, Field
from typing import Optional, Literal
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv

load_dotenv()

True

In [50]:
model= ChatMistralAI(model="mistral-medium-2508")

In [ ]:
class review(BaseModel): #schema
    
    summary: str= Field(description="A one sentence summary of the review")
    sentiment: Literal['achha review', 'bura review']= Field(description="return the sentiment of the review")
    key_themes: list[str]= Field(description="Write all the key themes discussed in the review in a list.")
    pros: Optional[list[str]]= Field(default= None, description="Write all the pros in a list")
    cons: Optional[list[str]]= Field(default= None, description="Write all the cons in a list")
    overall_rating: int= Field(description="On a scale of 1 to 10, write the overall rating of the product")
    name: Optional[str]= Field(default= None, description= "Write the name of the reviewer if present")

    
structured_model= model.with_structured_output(review)

review= """ 
After using the Nova X10 Pro as my primary device for the past two weeks, it is clear that the mid-range smartphone market has a new top contender. Priced at $599, the X10 Pro promises high-end features without the flagship price tag. It generally delivers on that promise, boasting an incredible display and lightning-fast performance, but it isn't perfect.
Here is my in-depth review of the Nova X10 Pro.
Design and Build Quality: Sleek but Slippery
The X10 Pro features a matte glass back that resists fingerprints, surrounded by an aluminum frame that feels very premium in hand. It is surprisingly light at 185g, but the matte finish makes it incredibly slippery—I dropped it twice in the first week.
Pros: IP68 water/dust resistance, satisfying clicky power button.
Cons: Very slippery, prone to scratches on the camera bump.
Display: A Visual Treat
The standout feature is the 6.7-inch AMOLED screen with a 120Hz adaptive refresh rate. Colors are vibrant, and the brightness peaks at 2,000 nits, making it perfect for outdoor viewing in direct sunlight. Watching HDR content on this phone is a joy, as the bezels are almost non-existent.
Performance: Gaming Powerhouse
Powered by the Snapdragon 8 Gen 4 chipset and 12GB of RAM, this phone flies. I ran Genshin Impact at max settings with minimal lag or overheating, thanks to the improved internal vapor chamber cooling system. Multitasking is smooth, and apps stay loaded in the background for a long time.
Camera Performance: Great Daytime, Decent Nighttime
The 50MP main sensor produces clean, sharp, and color-accurate photos in daylight. The portrait mode does an excellent job with edge detection.
However, in low-light situations, the noise reduction kicks in too heavily, leading to a "watercolor" effect on fine details. The 8MP ultrawide is decent but lacks the sharpness of the main lens.
Verdict: Excellent for social media, average for serious photography.
Battery Life and Charging
The 5,000mAh battery consistently lasted me through a full day of heavy usage, usually leaving me with 20% by bedtime. The 100W fast charging is phenomenal, getting me from 0% to 100% in just under 25 minutes.
"""

result= structured_model.invoke(review)

print(result.summary)
print(result.sentiment)
print(result.key_themes)
print(result.pros)
print(result.cons)
print(result.overall_rating)
print(result.name)

The Nova X10 Pro is a strong mid-range contender with a stunning display, powerful performance, and fast charging, but it has some drawbacks like a slippery design and average low-light camera performance.
achha review
['Design and Build Quality', 'Display', 'Performance', 'Camera Performance', 'Battery Life and Charging']
['IP68 water/dust resistance', 'Premium matte glass and aluminum build', '6.7-inch AMOLED display with 120Hz adaptive refresh rate', 'Vibrant colors and 2,000 nits brightness', 'Snapdragon 8 Gen 4 chipset with 12GB RAM for smooth performance', 'Excellent cooling system for gaming', '50MP main sensor with great daylight photography', '5,000mAh battery with all-day life', '100W fast charging (0% to 100% in 25 minutes)']
['Slippery design, prone to drops', 'Camera bump prone to scratches', 'Low-light camera performance suffers from heavy noise reduction', 'Ultrawide lens lacks sharpness']
9
None


# Structured Output using json schema

In [54]:
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv

load_dotenv()

True

In [55]:
model= ChatMistralAI(model="mistral-medium-2508")

In [62]:
#schema
json_schema= {
  "title": "review",
  "type": "object",
  "properties": {
    "summary": {
      "type": "string",
      "description": "A one sentence summary of the review"
    },
    "sentiment": {
      "type": "string",
      "enum": ["achha review", "bura review"],
      "description": "Return the sentiment of the review"
    },
    "key_themes": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Write all the key themes discussed in the review in a list."
    },
    "pros": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write all the pros in a list"
    },
    "cons": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write all the cons in a list"
    },
    "overall_rating": {
      "type": "integer",
      "minimum": 1,
      "maximum": 10,
      "description": "On a scale of 1 to 10, write the overall rating of the product"
    },
    "name": {
      "type": ["string", "null"],
      "description": "Write the name of the reviewer if present"
    }
  },
  "required": [
    "summary",
    "sentiment",
    "key_themes",
    "overall_rating"
  ],
  "additionalProperties": False
}
    
structured_model= model.with_structured_output(json_schema)

review= """ 
After using the Nova X10 Pro as my primary device for the past two weeks, it is clear that the mid-range smartphone market has a new top contender. Priced at $599, the X10 Pro promises high-end features without the flagship price tag. It generally delivers on that promise, boasting an incredible display and lightning-fast performance, but it isn't perfect.
Here is my in-depth review of the Nova X10 Pro.
Design and Build Quality: Sleek but Slippery
The X10 Pro features a matte glass back that resists fingerprints, surrounded by an aluminum frame that feels very premium in hand. It is surprisingly light at 185g, but the matte finish makes it incredibly slippery—I dropped it twice in the first week.
Pros: IP68 water/dust resistance, satisfying clicky power button.
Cons: Very slippery, prone to scratches on the camera bump.
Display: A Visual Treat
The standout feature is the 6.7-inch AMOLED screen with a 120Hz adaptive refresh rate. Colors are vibrant, and the brightness peaks at 2,000 nits, making it perfect for outdoor viewing in direct sunlight. Watching HDR content on this phone is a joy, as the bezels are almost non-existent.
Performance: Gaming Powerhouse
Powered by the Snapdragon 8 Gen 4 chipset and 12GB of RAM, this phone flies. I ran Genshin Impact at max settings with minimal lag or overheating, thanks to the improved internal vapor chamber cooling system. Multitasking is smooth, and apps stay loaded in the background for a long time.
Camera Performance: Great Daytime, Decent Nighttime
The 50MP main sensor produces clean, sharp, and color-accurate photos in daylight. The portrait mode does an excellent job with edge detection.
However, in low-light situations, the noise reduction kicks in too heavily, leading to a "watercolor" effect on fine details. The 8MP ultrawide is decent but lacks the sharpness of the main lens.
Verdict: Excellent for social media, average for serious photography.
Battery Life and Charging
The 5,000mAh battery consistently lasted me through a full day of heavy usage, usually leaving me with 20% by bedtime. The 100W fast charging is phenomenal, getting me from 0% to 100% in just under 25 minutes.
"""

result= structured_model.invoke(review)

print(result['summary'])
print(result['sentiment'])
print(result['key_themes'])
print(result['pros'] if 'pros' in result else None)
print(result['cons'] if 'cons' in result else None)
print(result['overall_rating'])
print(result['name'] if 'name' in result else None)

The Nova X10 Pro is a strong mid-range contender with a stunning display, powerful performance, and fast charging, though it has some drawbacks like a slippery design and average low-light camera performance.
achha review
['Design and Build Quality', 'Display', 'Performance', 'Camera Performance', 'Battery Life and Charging']
['IP68 water/dust resistance', 'Sleek and premium aluminum frame', '6.7-inch AMOLED screen with 120Hz adaptive refresh rate', 'Vibrant colors and 2,000 nits brightness', 'Snapdragon 8 Gen 4 chipset and 12GB RAM for smooth performance', 'Excellent cooling system for gaming', '50MP main camera with great daylight performance', '5,000mAh battery with all-day life', '100W fast charging (0% to 100% in 25 minutes)']
['Slippery matte glass back', 'Prone to scratches on the camera bump', 'Low-light camera performance suffers from heavy noise reduction', 'Ultrawide camera lacks sharpness']
9
None
